In [1]:
!pip install ultralytics roboflow opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 118.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13


In [2]:
import os
HOME = os.getcwd()

In [3]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.5/112.6 GB disk)


In [87]:
# import the dataset from roboflow
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="bgZz4XsPKrewZX7Qhbab")
project = rf.workspace("ahmeds-workspace-slwao").project("bike_fit")
version = project.version(4)
dataset = version.download("yolov8")




loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to bike_fit-4 in yolov8:: 100%|██████████| 65/65 [00:00<00:00, 4426.10it/s]


In [88]:
from ultralytics import YOLO

model  = YOLO('yolo11n-pose.pt')

results = model.train(data="bike_fit-4/data.yaml", epochs=200, imgsz=640  )

Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bike_fit-4/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, 

In [84]:
import numpy as np
import math

def calculate_angle(A, B, C):

    A = np.array(A) # bottom point
    B = np.array(B) # center point
    C = np.array(C) # upper point

    # we calculate the vectors of BA  , BC
    BA = A - B
    BC = C - B

    dot_product = np.dot(BA, BC)

    magnitude_BA = np.linalg.norm(BA)
    magnitude_BC = np.linalg.norm(BC)

    cos_angle = dot_product / (magnitude_BA * magnitude_BC)

    cos_angle = np.clip(cos_angle, -1.0, 1.0)

    angle = np.arccos(cos_angle)

    angle = np.degrees(angle)

    angle1 = math.degrees(
    math.atan2(A[1] - B[1], A[0] - B[0])
    )

    angle2 = math.degrees(
    math.atan2(C[1] - B[1], C[0] - B[0])
    )
    radius = 40

    start_angle = int(angle1)
    end_angle = int(angle2)

    return angle , start_angle , end_angle , radius

In [85]:
def feedback(angles) :
  # some feedback based on angle from some web search
  notes = {}
  elpow_angle , shoulder_ange  , hip_angle , knee_angle = angles
  if int(elpow_angle) in range(150 , 170) :
    notes['Elbow angle'] = 'within range'
  elif int(elpow_angle) < 150 :
    notes['Elbow angle'] = 'Try longer Stem'
  elif int(elpow_angle) > 170 :
    notes['Elbow angle'] = 'Try shorter Stem'

  if int(shoulder_ange) in range(70 , 95) :
    notes['Shoulder angle'] = 'within range'
  elif int(shoulder_ange) < 70 :
    notes['Shoulder angle'] = 'Try moving the seat further back'
  elif int(shoulder_ange) > 95 :
    notes['Shoulder angle'] = 'Try moving the seat forword'

  if int(hip_angle) in range(60 , 125) :
    notes['Hip angle'] = 'within range'
  elif int(shoulder_ange) < 60 :
    notes['Hip angle'] = 'Try elevate the seat'
  elif int(shoulder_ange) > 125 :
    notes['Hip angle'] = 'Try lowering the seat '

  if int(knee_angle) in range(60 , 150) :
    notes['Knee angle'] = 'within range'
  elif int(shoulder_ange) < 60 :
    notes['Knee angle'] = 'Try elevate the seat'
  elif int(shoulder_ange) > 125 :
    notes['Knee angle'] = 'Try lowering the seat '

  return notes

In [95]:

import cv2
img = cv2.imread("/content/frame_760.jpg")
# Input and output video paths
input_video = "bike.mp4"
output_video = "p_final.mp4"

# Open input video
cap = cv2.VideoCapture(input_video)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

# we take the line between every consecutive two points
lines = [
    (0, 1), (1, 2),
    (2, 3), (3, 4),
    (4, 5),
]
# we take the angle between sets of three points
angles = [
    (0 , 1, 2) ,
    (1,2,3) ,
    (2,3,4) ,
    (3,4,5)
]
while True:
    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame)
    for result in results:

      keypoints = result.keypoints.xy.cpu().numpy()

      for point_set in keypoints:

          # Draw keypoints
          for i, (x, y) in enumerate(point_set):

              x = int(x)
              y = int(y)

              cv2.circle(frame, (x, y), 5, (0, 255, 0), -1)


          for p1, p2 in lines:

              x1, y1 = point_set[p1]
              x2, y2 = point_set[p2]

              cv2.line(
                  frame,
                  (int(x1), int(y1)),
                  (int(x2), int(y2)),
                  (0, 0, 255),
                  2
              )
          Angles = []
          for A1 , A2 ,A3 in angles :
              angle , start_angle , end_angle , radius = calculate_angle(point_set[A1] , point_set[A2] , point_set[A3])
              Angles.append(angle)
              cv2.ellipse(
                  frame,
                  (int(point_set[A2][0]) , int(point_set[A2][1])),
                  (radius, radius),
                  0,
                  start_angle,
                  end_angle,
                  (255, 0, 0),
                  2
                )
              cv2.putText(
                frame,
                f"{int(angle)} o",
                (int(point_set[A2][0]) +20 , int(point_set[A2][1])-20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0, 0, 255),
                2
              )
          notes = feedback(Angles)
          for  i , (A , n) in enumerate(notes.items()) :
            cv2.putText(
                frame,
                f"{A}: {n}",
                ((80) , (50 + 20*i)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0, 0, 255),
                2
              )

          out.write( frame)


cap.release()
out.release()
cv2.destroyAllWindows()


0: 416x640 1 points, 23.1ms
Speed: 6.7ms preprocess, 23.1ms inference, 2.2ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 points, 21.0ms
Speed: 8.6ms preprocess, 21.0ms inference, 1.8ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 points, 34.6ms
Speed: 6.5ms preprocess, 34.6ms inference, 4.1ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 points, 37.4ms
Speed: 7.1ms preprocess, 37.4ms inference, 2.0ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 points, 28.5ms
Speed: 9.0ms preprocess, 28.5ms inference, 3.8ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 points, 24.5ms
Speed: 6.6ms preprocess, 24.5ms inference, 3.8ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 points, 30.8ms
Speed: 5.8ms preprocess, 30.8ms inference, 4.9ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 points, 26.7ms
Speed: 7.9ms preprocess, 26.7ms inference, 3.8ms postprocess per image at shape (1, 3, 41

error: OpenCV(4.10.0) /io/opencv/modules/highgui/src/window.cpp:1295: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvDestroyAllWindows'
